In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.s4lake_silver"
GOLD = "workspace.s4lake_gold"

def salvar(df, schema, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{schema}.{tabela}")

fato_titulos = spark.table(f"{GOLD}.fato_titulos")

no_periodo = F.col("data_documento").between(F.date_sub(F.col("data_corte"), 89), F.col("data_corte"))

base = fato_titulos.agg(
    F.max("data_corte").alias("data_corte"),

    F.sum(F.when(F.col("status") == "aberto", F.col("valor"))).alias("carteira_aberta"),

    F.sum(F.when(F.col("situacao") == "vencido", F.col("valor"))).alias("carteira_vencida"),

    F.sum(F.when(F.col("status") == "pago", F.lit(1))).alias("qtd_pagos"),

    F.sum(F.when(F.col("situacao") == "pago com atraso", 
    F.lit(1))).alias("qtd_pagos_com_atraso"),

    F.sum(F.when(F.col("status") == "pago", F.greatest(F.col("dias_atraso"), F.lit(0)))).alias("soma_atraso"),

    F.sum(F.when(F.col("status") == "pago", F.col("valor"))).alias("soma_valor_pagos"),

    F.sum(F.when(F.col("status") == "pago", F.col("valor") * F.greatest(F.col("dias_atraso"), F.lit(0)))).alias("soma_valor_x_atraso"),

    F.sum(F.when(no_periodo, F.col("valor"))).alias("faturamento_90d"),

    F.sum(F.when(no_periodo, F.col("valor") * F.col("prazo_dias"))).alias("soma_valor_x_prazo_90d"),

)

kpis_gerais = (
    base
    .withColumn("pct_vencido", F.round(F.col("carteira_vencida") / F.col("carteira_aberta") * 100, 2))

    .withColumn("pct_pagos_com_atraso", F.round(F.col("qtd_pagos_com_atraso") / F.col("qtd_pagos") * 100, 2))

    .withColumn("atraso_medio_simples", F.round(F.col("soma_atraso") / F.col("qtd_pagos"), 2))

    .withColumn("atraso_medio_ponderado", F.round(F.col("soma_valor_x_atraso") / F.col("soma_valor_pagos"), 2))

    .withColumn("dso", F.round(F.col("carteira_aberta") / F.col("faturamento_90d") * 90, 2))

    .withColumn("prazo_medio_ponderado", F.round(F.col("soma_valor_x_prazo_90d") / F.col("faturamento_90d"), 2))

    .withColumn("dias_alem_prazo", F.round(F.col("dso") - F.col("prazo_medio_ponderado"), 2))
)

salvar(kpis_gerais, GOLD, "kpis_gerais")

In [0]:
spark.sql(f"""
    SELECT pct_vencido, pct_pagos_com_atraso, atraso_medio_simples, atraso_medio_ponderado
    FROM {GOLD}.kpis_gerais
""").show()

spark.sql(f"""
    SELECT dso, prazo_medio_ponderado 
    FROM {GOLD}.kpis_gerais
""").show()

In [0]:
%sql
SELECT dso, prazo_medio_ponderado, dias_alem_prazo
FROM workspace.s4lake_gold.kpis_gerais

In [0]:
%sql
COMMENT ON TABLE workspace.s4lake_gold.kpis_gerais IS
'Painel de indicadores de contas a receber da Horizonte Embalagens. Uma linha resume todos os títulos a receber na data de corte do pipeline (constante DATA_CORTE do notebook 06): títulos em aberto são medidos contra essa data, e não contra a data de hoje. Valores em R$ são brutos (com impostos, campo DMBTR do SAP). Percentuais estão na escala de 0 a 100. Colunas soma_ e qtd_ são a matéria-prima dos indicadores, mantidas para que qualquer indicador possa ser recalculado.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN carteira_aberta COMMENT
'Carteira em aberto: soma do valor bruto, em R$, dos títulos ainda não pagos na data de corte, vencidos ou a vencer.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN carteira_vencida COMMENT
'Carteira vencida: soma do valor bruto, em R$, dos títulos não pagos cujo vencimento já passou na data de corte. É uma parte da carteira_aberta.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN qtd_pagos COMMENT
'Quantidade de títulos já pagos (compensados) até a data de corte.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN qtd_pagos_com_atraso COMMENT
'Quantidade de títulos pagos depois do vencimento (pelo menos 1 dia de atraso).';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN soma_atraso COMMENT
'Soma dos dias de atraso dos títulos pagos, em dias. Pagamentos antecipados contam como zero, e não como atraso negativo.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN soma_valor_pagos COMMENT
'Soma do valor bruto, em R$, dos títulos pagos até a data de corte.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN soma_valor_x_atraso COMMENT
'Soma de valor x dias de atraso dos títulos pagos, em reais-dia (dinheiro parado fora do caixa por causa do atraso). Pagamentos antecipados contam como zero dias. Base do atraso_medio_ponderado.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN faturamento_90d COMMENT
'Faturamento dos últimos 90 dias: soma do valor bruto, em R$, dos títulos de faturas emitidas nos 90 dias até a data de corte, incluindo a própria data. Base do DSO.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN soma_valor_x_prazo_90d COMMENT
'Soma de valor x prazo de pagamento concedido (em dias) das faturas dos últimos 90 dias. Base do prazo_medio_ponderado.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN pct_vencido COMMENT
'Percentual da carteira em aberto que já está vencida, em valor (escala de 0 a 100): carteira_vencida / carteira_aberta x 100. Retrato da carteira na data de corte; quanto maior, pior.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN pct_pagos_com_atraso COMMENT
'Percentual dos títulos pagos que foram pagos com atraso, em quantidade (escala de 0 a 100): qtd_pagos_com_atraso / qtd_pagos x 100. Mede o comportamento de pagamento ao longo do histórico; quanto maior, pior.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN atraso_medio_simples COMMENT
'Atraso médio dos títulos pagos, em dias, com o mesmo peso para cada título: soma_atraso / qtd_pagos. Pagamentos antecipados contam como zero. Considera só títulos pagos, então subestima o problema: títulos que nunca foram pagos ficam de fora.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN atraso_medio_ponderado COMMENT
'Atraso médio dos títulos pagos, em dias, ponderado pelo valor: soma_valor_x_atraso / soma_valor_pagos. Mede o impacto do atraso no caixa (cada real pesa igual). Se for maior que o atraso_medio_simples, as faturas maiores atrasam mais.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN dso COMMENT
'DSO (Days Sales Outstanding), em dias: quantos dias de vendas estão parados no contas a receber. Fórmula: carteira_aberta / faturamento_90d x 90. Deve ser lido junto com o prazo_medio_ponderado, porque o DSO nunca fica abaixo do prazo concedido. Quanto maior, pior.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN prazo_medio_ponderado COMMENT
'Prazo médio de pagamento concedido aos clientes, em dias, ponderado pelo valor das faturas dos últimos 90 dias: soma_valor_x_prazo_90d / faturamento_90d. É a referência para ler o DSO.';

ALTER TABLE workspace.s4lake_gold.kpis_gerais ALTER COLUMN dias_alem_prazo COMMENT
'Dias além do prazo, em dias: dso - prazo_medio_ponderado. Quanto o recebimento demora a mais do que o prazo concedido. É a medida certa para comparar grupos com prazos diferentes. Não confundir com o Average Days Delinquent de livro (DSO menos Best Possible DSO), que é outra medida.';

In [0]:
%sql
DESCRIBE TABLE workspace.s4lake_gold.kpis_gerais